## imports and settings

In [ ]:
# use this after changes in libtoolkit

%load_ext autoreload
%autoreload 2

In [ ]:
import libtoolkit

import pandas as pd
import numpy as np
import keyring, requests, json, re, time, os, copy, yaml
from requests import Session
from requests.auth import HTTPBasicAuth
from lxml import etree
from pathlib import Path
from io import StringIO
from datetime import datetime
from collections import Counter

# datetime in format: 20250123
dateNow = datetime.now().strftime("%Y%m%d")

# expand
baseP = Path("C:/Users/onb1340/dev-data/...")     


pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_columns', None)
#pd.set_option('display.max_rows', None)

In [8]:
baseP.exists()

True

## helpers

## base analyses

## pandas cheat sheet

#### GroupBy & Custom Aggregations

In [19]:
import pandas as pd
import numpy as np

# example DataFrame
exampleDF = pd.DataFrame({
    "MMS-ID": ["101", "102", "103", "104", None, None],
    "Title": ["Book A", "Book B", "Book C", "Book D", "Book E", "Book F"],
    "Barcode": ["BC_01", "BC_02", "BC_03", None, "BC_05", "BC_06"],
    "Price": [15, 20, 15, 35, 20, 40],
    "Status": ["Available", "Loaned", "Available", "Available", "Lost", "Lost"],
    "Library": ["ZNEU", "ZNEU", "ZALT", "ZESP", "ZHAN", "ZMUS"]
})

# aggregation showcase
aggDF = exampleDF.groupby(
    ["Library", "Status"], 
    dropna=False        # keeps rows where grouping keys are NaN
).agg(
    # named aggregations: (Column_to_aggregate, Function_to_apply)
    BarcodeCount=("Barcode", "count"),                  # Counts non-NaN values
    TotalRows=("Barcode", "size"),                      # Counts ALL rows (including NaNs)
    UniqueStatuses=("Status", "nunique"),               # Number of unique values
    MMSList=("MMS-ID", lambda x: list(x)),              # Custom lambda to make a list
    TitleList=("Title", lambda x: list(x)),
    AveragePrice=("Price", "mean"),                     # Basic math mean
    PriceMode=("Price", lambda x: x.mode()[0] if not x.mode().empty else None)    # Most frequent value, makes no sense here but could use
)#.reset_index()     # comment this for grouped indices

aggDF

BarcodeCount  TotalRows  UniqueStatuses MMSList TitleList  \
Library Status                                                                 
ZALT    Available             1          1               1   [103]  [Book C]   
ZESP    Available             0          1               1   [104]  [Book D]   
ZHAN    Lost                  1          1               1   [nan]  [Book E]   
ZMUS    Lost                  1          1               1   [nan]  [Book F]   
ZNEU    Available             1          1               1   [101]  [Book A]   
        Loaned                1          1               1   [102]  [Book B]   

                   AveragePrice  PriceMode  
Library Status                              
ZALT    Available          15.0         15  
ZESP    Available          35.0         35  
ZHAN    Lost               20.0         20  
ZMUS    Lost               40.0         40  
ZNEU    Available          15.0         15  
        Loaned             20.0         20

#### Column Transformations

##### 1-to-N Column Transformation (One Column $\rightarrow$ Multiple Columns)

In [ ]:
# example DataFrame
exampleDF = pd.DataFrame({
    "MMS-ID": ["101", "102", "103", "104", None, None],
    "Title": ["Book A", "Book B", "Book C", "Book D", "Book E", "Book F"],
    "Barcode": ["BC_01", "BC_02", "BC_03", None, "BC_05", "BC_06"],
    "Price": [15, 20, 15, 35, 20, 40],
    "Status": ["Available", "Loaned", "Available", "Available", "Lost", "Lost"],
    "Library": ["ZNEU", "ZNEU", "ZALT", "ZESP", "ZHAN", "ZMUS"]
})

# important! re-index if df is combined
exampleDF = exampleDF.reset_index(drop=True)

# dummy function simulating your getIdInfoSRU function
def getMockIdInfo(mmsId):
    if pd.isna(mmsId):
        return [None, None, "Error: Missing ID"]
    
    # returns a list or dict that will become row values
    return [f"Date_{mmsId}", f"Author_{mmsId}", "Success"]

# general template
# 1. apply function to the source column (returns a Series of lists/dicts)
# 2. use .apply(pd.Series) to split the lists into individual columns
expandedColsDF = exampleDF["MMS-ID"].apply(getMockIdInfo).apply(pd.Series)

# 3. rename the newly generated columns
expandedColsDF.columns = ["enrichDate", "enrichAuthor", "fetchStatus"]

# 4. join back to the original DataFrame
exampleDF = exampleDF.join(expandedColsDF)


exampleDF

,MMS-ID,Title,Barcode,Price,Status,Library,enrichDate,enrichAuthor,fetchStatus
0,101,Book A,BC_01,15,Available,ZNEU,Date_101,Author_101,Success
1,102,Book B,BC_02,20,Loaned,ZNEU,Date_102,Author_102,Success
2,103,Book C,BC_03,15,Available,ZALT,Date_103,Author_103,Success
3,104,Book D,NaN,35,Available,ZESP,Date_104,Author_104,Success
4,NaN,Book E,BC_05,20,Lost,ZHAN,NaN,NaN,Error: Missing ID
5,NaN,Book F,BC_06,40,Lost,ZMUS,NaN,NaN,Error: Missing ID


##### N-to-1 Column Transformation (Multiple Columns $\rightarrow$ One Column)

In [ ]:
# example DataFrame
exampleDF = pd.DataFrame({
    "MMS-ID": ["101", "102", "103", "104", None, None],
    "Title": ["Book A", "Book B", "Book C", "Book D", "Book E", "Book F"],
    "Barcode": ["BC_01", "BC_02", "BC_03", None, "BC_05", "BC_06"],
    "Price": [15, 20, 15, 35, 20, 40],
    "Status": ["Available", "Loaned", "Available", "Available", "Lost", "Lost"],
    "Library": ["ZNEU", "ZNEU", "ZALT", "ZESP", "ZHAN", "ZMUS"]
})

# approach A: Using .apply() with axis=1 (best for complex/custom logic)
def determineHandlingStatus(row):
    
    if row["Price"] > 30 and row["Status"] == "Lost":
        return "High Priority Investigation"
    
    elif row["Status"] == "Available":
        return "Shelved"
    
    else:
        return "Standard Check"

exampleDF["HandlingAction"] = exampleDF.apply(determineHandlingStatus, axis=1)

# approach B: using np.select() (highly scalable & much faster for standard conditions)
conditions = [
    (exampleDF["Price"] > 30) & (exampleDF["Status"] == "Lost"),
    (exampleDF["Status"] == "Available")
]
choices = ["High Priority Investigation", "Shelved"]

exampleDF["HandlingActionFast"] = np.select(conditions, choices, default="Standard Check")

exampleDF

,MMS-ID,Title,Barcode,Price,Status,Library,HandlingAction,HandlingActionFast
0,101,Book A,BC_01,15,Available,ZNEU,Shelved,Shelved
1,102,Book B,BC_02,20,Loaned,ZNEU,Standard Check,Standard Check
2,103,Book C,BC_03,15,Available,ZALT,Shelved,Shelved
3,104,Book D,NaN,35,Available,ZESP,Shelved,Shelved
4,NaN,Book E,BC_05,20,Lost,ZHAN,Standard Check,Standard Check
5,NaN,Book F,BC_06,40,Lost,ZMUS,High Priority Investigation,High Priority Investigation


##### M-to-N Column Transformation (Multiple Columns $\rightarrow$ Multiple Columns)

In [17]:
exampleDF = pd.DataFrame({
    "MMS-ID": ["101", "102", "103", "104", None, None],
    "Title": ["Book A", "Book B", "Book C", "Book D", "Book E", "Book F"],
    "Barcode": ["BC_01", "BC_02", "BC_03", None, "BC_05", "BC_06"],
    "Price": [15, 20, 15, 35, 20, 40],
    "Status": ["Available", "Loaned", "Available", "Available", "Lost", "Lost"],
    "Library": ["ZNEU", "ZNEU", "ZALT", "ZESP", "ZHAN", "ZMUS"]
})

# define a function that accepts the whole row and returns a pd.Series
def complexRowProcessor(row):
    
    # Process multiple inputs
    price = row["Price"]
    status = row["Status"]
    
    # calculate multiple outputs
    tax = price * 0.10
    isRisk = True if (status == "Lost" and price > 20) else False
    
    # return as a Pandas Series with the index set to your new column names
    return pd.Series([tax, isRisk], index=["TaxCalculated", "IsFinancialRisk"])

# apply across axis=1 and assign directly to new columns
newCols = ["TaxCalculated", "IsFinancialRisk"]
exampleDF[newCols] = exampleDF.apply(complexRowProcessor, axis=1)

exampleDF

,MMS-ID,Title,Barcode,Price,Status,Library,TaxCalculated,IsFinancialRisk
0,101,Book A,BC_01,15,Available,ZNEU,1.5,False
1,102,Book B,BC_02,20,Loaned,ZNEU,2.0,False
2,103,Book C,BC_03,15,Available,ZALT,1.5,False
3,104,Book D,NaN,35,Available,ZESP,3.5,False
4,NaN,Book E,BC_05,20,Lost,ZHAN,2.0,False
5,NaN,Book F,BC_06,40,Lost,ZMUS,4.0,True


#### Combining Data Explicitly (merge vs concat)

##### setup example dfs

In [ ]:
import pandas as pd

# Master DataFrame (e.g., primary catalog)
dfLeft = pd.DataFrame({
    "MMS Id": ["101", "102", "103", "104"],
    "Title": ["Book A", "Book B", "Book C", "Book D"],
    "Language": ["en", "de", "fr", "en"]
})

# enrichment DataFrame (e.g., usage or electronic inventory statistics)
dfRight = pd.DataFrame({
    "MMS Id": ["102", "104", "105"],
    "UsageCount": [45, 12, 98],
    "Supplier": ["Vendor X", "Vendor Y", "Vendor X"]
})

##### Explicit Left Join (pd.merge + indicator=True)

In [23]:
# standard left join template
mergedLeftDF = pd.merge(
    dfLeft, 
    dfRight, 
    on="MMS Id", 
    how="left", 
    indicator=True,       # creates the '_merge' column showing 'left_only', 'right_only', or 'both'
    validate="1:1"        # Enforces that MMS Id is unique in BOTH dfs. Options: '1:1', '1:m', 'm:1'
)

mergedLeftDF

# mergedLeftDF["_merge"].value_counts()     # helpful

,MMS Id,Title,Language,UsageCount,Supplier,_merge
0,101,Book A,en,NaN,NaN,left_only
1,102,Book B,de,45.0,Vendor X,both
2,103,Book C,fr,NaN,NaN,left_only
3,104,Book D,en,12.0,Vendor Y,both


##### Outer Join (For finding mismatches)

In [24]:
# outer join to see the full picture
mergedOuterDF = pd.merge(
    dfLeft, 
    dfRight, 
    on="MMS Id", 
    how="outer", 
    indicator=True
)

mergedOuterDF

,MMS Id,Title,Language,UsageCount,Supplier,_merge
0,101,Book A,en,NaN,NaN,left_only
1,102,Book B,de,45.0,Vendor X,both
2,103,Book C,fr,NaN,NaN,left_only
3,104,Book D,en,12.0,Vendor Y,both
4,105,NaN,NaN,98.0,Vendor X,right_only


##### Stacking Rows Vertically (pd.concat)

In [26]:
# setup two identical batches
batchJanDF = pd.DataFrame({"MMS-ID": ["101", "102"], "Views": [10, 20]})
batchFebDF = pd.DataFrame({"MMS-ID": ["103", "104"], "Views": [15, 30]})

# stack them vertically (axis=0 is default)
combinedYearDF = pd.concat(
    [batchJanDF, batchFebDF], 
    axis=0, 
    ignore_index=True   # Prevents duplicate index numbers (0, 1, 0, 1 becomes 0, 1, 2, 3)
)

combinedYearDF

,MMS-ID,Views
0,101,10
1,102,20
2,103,15
3,104,30


#### Modifying Specific Cells Safely

##### Setup Example Data

In [ ]:
import pandas as pd
import numpy as np

exampleDF = pd.DataFrame({
    "MMS-ID": ["101", "102", "103", "104"],
    "Title": ["Book A", "Book B", "Book C", "Book D"],
    "Status": ["Available", "Loaned", "Lost", "Available"]
})

In [ ]:
exampleDF

,Title,Status
MMS-ID,,
101,Book A,Available
102,Book B,Loaned
103,Book C,Lost
104,Book D,Available


##### .loc with a conditional row filter

In [ ]:
# Syntax: df.loc[ row_condition, "Column_Name" ] = new_value

exampleDF.loc[exampleDF["MMS-ID"] == "103", "Status"] = "Available"

In [ ]:
exampleDF

,MMS-ID,Title,Status
0,101,Book A,Available
1,102,Book B,Loaned
2,103,Book C,Available
3,104,Book D,Available


##### .iloc + Dynamic Column Index

In [ ]:
# find the integer position of the "Status" column
colPos = exampleDF.columns.get_loc("Status")  # returns 2

# use .iloc with the row number and the calculated column position
# change the 3rd row (index 2) in the Status column
exampleDF.iloc[2, colPos] = "Available"

In [ ]:
exampleDF

,MMS-ID,Title,Status
0,101,Book A,Available
1,102,Book B,Loaned
2,103,Book C,Available
3,104,Book D,Available


##### The Precision Strike: .loc[row_label, col_label]

In [ ]:
# setting MMS-ID as the index makes label lookup incredibly powerful
exampleDF.set_index("MMS-ID", inplace=True)

In [ ]:
# Syntax: df.loc[row_index_label, column_name] = new_value

# Change the Status of Book 103 to "Available"
exampleDF.loc["103", "Status"] = "Available"

In [ ]:
exampleDF

,Title,Status
MMS-ID,,
101,Book A,Available
102,Book B,Loaned
103,Book C,Available
104,Book D,Available


##### The Positional Fix: .iloc[row_position, col_position]

In [ ]:
# Syntax: df.iloc[row_integer_position, column_integer_position] = new_value

# Overwrite the Title of the very first row (index 0) to "Updated Book A"
exampleDF.iloc[0, 0] = "Updated Book A"

In [ ]:
exampleDF

,Title,Status
MMS-ID,,
101,Updated Book A,Available
102,Book B,Loaned
103,Book C,Available
104,Book D,Available


#### Drop Duplicates

In [1]:
import pandas as pd

data = {
    'user': ['Alice', 'Bob', 'Alice', 'Charlie', 'Alice'],
    'score': [10, 20, 30, 40, 50]
}
exampleDF = pd.DataFrame(data)

In [2]:
exampleDF

,user,score
0,Alice,10
1,Bob,20
2,Alice,30
3,Charlie,40
4,Alice,50


In [ ]:
# keep the first record (default)
# "subset" specifies the column to check for duplicates

firstDF = exampleDF.drop_duplicates(subset=["user"], keep="first")
firstDF

,user,score
0,Alice,10
1,Bob,20
3,Charlie,40


In [5]:
# keep the last record

lastDF = exampleDF.drop_duplicates(subset=["user"], keep="last")
lastDF

,user,score
1,Bob,20
3,Charlie,40
4,Alice,50


In [6]:
# drop all duplicates entirely

noneDF = exampleDF.drop_duplicates(subset=["user"], keep=False)
noneDF

,user,score
1,Bob,20
3,Charlie,40


In [7]:
# don't just drop the first or last duplicate row

# sort by score descending so the highest score is at the top
sortedDF = exampleDF.sort_values(by="score", ascending=False)

# drop duplicates keeping "first" (which is now the highest score)
highestDF = sortedDF.drop_duplicates(subset=["user"], keep="first")

highestDF

,user,score
4,Alice,50
3,Charlie,40
1,Bob,20


#### Excel Raw String Diagnostics (NaNs & Duplicates)

In [37]:
import io
import pandas as pd
import numpy as np

# ==========================================
# STEP 0: SIMULATE THE "IN THE WILD" EXCEL FILE
# ==========================================
# We create a real Excel file byte-stream in memory
excelStream = io.BytesIO()
rawData = pd.DataFrame({
    "MMS-ID": ["101", "102 ", "101", "103", "nan", None],                   # Missing cell (None)
    "Title": ["Book A", "Book B", "Book A", "Book C", "Book D", "Book E"],
    "Barcode": ["BC01", "BC02", "BC01", "  ", "BC04", "BC05"]               # Blank space cell
})

with pd.ExcelWriter(excelStream, engine="openpyxl") as writer:
    rawData.to_excel(writer, index=False)
excelStream.seek(0) # Reset stream pointer to the beginning

# ==========================================
# THE TEMPLATE STARTS HERE
# ==========================================

# read the Excel file forcing string evaluation
exampleDF = pd.read_excel(excelStream, dtype=str)

print("--- Step 1: Clean String Artifacts Safely ---")
# strip outer whitespace from columns, safely ignoring float NaNs
for col in exampleDF.columns:
    exampleDF[col] = exampleDF[col].str.strip()

# convert standard Excel/Pandas string clutter into genuine NaNs
# 'nan' covers the literal text "nan" typed into Excel, and the stringified version of empty cells
invalidStrings = ["", "nan", "NaN", "None", "null"]
exampleDF = exampleDF.replace(invalidStrings, np.nan)


print("\n--- Step 2: Identify Missing Data (NaNs) ---")
nanCounts = exampleDF.isna().sum()
nanPct = (exampleDF.isna().sum() / len(exampleDF) * 100).round(2)

nanReportDF = pd.DataFrame({"Missing Count": nanCounts, "Percentage (%)": nanPct})
print(nanReportDF)


print("\n--- Step 3: Identify & Isolate Duplicates ---")
totalDuplicates = exampleDF.duplicated().sum()
print(f"Total fully duplicated rows across the whole sheet: {totalDuplicates}")

# Isolate duplicate IDs side-by-side (ignoring the NaNs we just cleaned)
duplicateIdsDF = exampleDF[exampleDF.duplicated(subset=["MMS-ID"], keep=False) & exampleDF["MMS-ID"].notna()]

print("\nRows with duplicate MMS Ids:")
print(duplicateIdsDF.sort_values(by="MMS-ID"))

--- Step 1: Clean String Artifacts Safely ---

--- Step 2: Identify Missing Data (NaNs) ---
         Missing Count  Percentage (%)
MMS-ID               2           33.33
Title                0            0.00
Barcode              1           16.67

--- Step 3: Identify & Isolate Duplicates ---
Total fully duplicated rows across the whole sheet: 1

Rows with duplicate MMS Ids:
  MMS-ID   Title Barcode
0    101  Book A    BC01
2    101  Book A    BC01


In [30]:
exampleDF

,MMS-ID,Title,Barcode
0,101,Book A,BC01
1,102,Book B,BC02
2,101,Book A,BC01
3,103,Book C,
4,NaN,Book D,BC04
5,NaN,Book E,BC05


In [ ]:
# keep the duplicate row that is the most complete

exampleDF["_nonNullCount"] = exampleDF.notna().sum(axis=1)
exampleDF = exampleDF.sort_values("_nonNullCount", ascending=False)
exampleDF = exampleDF.drop_duplicates(subset=["MMS-ID"], keep="first").drop(columns=["_nonNullCount"])

#### Defensive Date Parsing & Timedeltas

##### Setup Example Data

In [38]:
import io
import pandas as pd
import numpy as np

# simulate an Excel file with messy string dates
excelStream = io.BytesIO()
rawData = pd.DataFrame({
    "MMS-ID": ["101", "102", "103", "104"],
    "CreationDate": ["2026-07-01", "05/12/2026", "2026-07-13 14:30:00", "not a date"],
    "ClosedDate": ["2026-07-10", "2026-05-20", "2026-07-15 09:00:00", None]
})

with pd.ExcelWriter(excelStream, engine="openpyxl") as writer:
    rawData.to_excel(writer, index=False)
excelStream.seek(0)

# read it in as strings
exampleDF = pd.read_excel(excelStream, dtype=str)

##### Date-Handling Template

In [39]:
print("--- Step 1: Defensive Date Parsing ---")
# use pd.to_datetime with errors='coerce'
# this forces unparseable strings (like "not a date") into NaT (Not a Time) instead of crashing
# format='mixed' allows Pandas to guess formats row-by-row (e.g., ISO vs US dates) safely

exampleDF["CreationDate"] = pd.to_datetime(exampleDF["CreationDate"], errors="coerce", format="mixed")
exampleDF["ClosedDate"] = pd.to_datetime(exampleDF["ClosedDate"], errors="coerce", format="mixed")

print(exampleDF.dtypes)
print(exampleDF[["CreationDate", "ClosedDate"]])


print("\n--- Step 2: Date Math & Time Differences (Timedeltas) ---")
# subtracting two datetimes results in a Pandas 'Timedelta' object
exampleDF["Duration"] = exampleDF["ClosedDate"] - exampleDF["CreationDate"]

# extract specific components from the Timedelta
exampleDF["DaysToClose"] = exampleDF["Duration"].dt.days             # whole days as integers
exampleDF["TotalHoursToClose"] = exampleDF["Duration"].dt.total_seconds() / 3600  # exact hours (float)

print(exampleDF[["Duration", "DaysToClose", "TotalHoursToClose"]])


print("\n--- Step 3: Extracting Properties (The .dt Accessor) ---")
# like .str helps with strings, .dt unlocks datetime properties
exampleDF["Year"] = exampleDF["CreationDate"].dt.year
exampleDF["Month"] = exampleDF["CreationDate"].dt.month
exampleDF["DayName"] = exampleDF["CreationDate"].dt.day_name()       # e.g., 'Monday'
exampleDF["IsWeekend"] = exampleDF["CreationDate"].dt.dayofweek >= 5  # True/False

print(exampleDF[["CreationDate", "Year", "DayName", "IsWeekend"]])


print("\n--- Step 4: Filtering by Dates ---")
# 1. filter using a string cutoff
julyRecordsDF = exampleDF[exampleDF["CreationDate"] >= "2026-07-01"]

# 2. filter using a dynamic window (e.g., records closed within the last 30 days)
# using pd.Timestamp.now() for the current execution time
thirtyDaysAgo = pd.Timestamp.now() - pd.Timedelta(days=30)
recentRecordsDF = exampleDF[exampleDF["ClosedDate"] >= thirtyDaysAgo]

print(recentRecordsDF)

--- Step 1: Defensive Date Parsing ---
MMS-ID                     str
CreationDate    datetime64[us]
ClosedDate      datetime64[us]
dtype: object
         CreationDate          ClosedDate
0 2026-07-01 00:00:00 2026-07-10 00:00:00
1 2026-05-12 00:00:00 2026-05-20 00:00:00
2 2026-07-13 14:30:00 2026-07-15 09:00:00
3                 NaT                 NaT

--- Step 2: Date Math & Time Differences (Timedeltas) ---
         Duration  DaysToClose  TotalHoursToClose
0 9 days 00:00:00          9.0              216.0
1 8 days 00:00:00          8.0              192.0
2 1 days 18:30:00          1.0               42.5
3             NaT          NaN                NaN

--- Step 3: Extracting Properties (The .dt Accessor) ---
         CreationDate    Year    DayName  IsWeekend
0 2026-07-01 00:00:00  2026.0  Wednesday      False
1 2026-05-12 00:00:00  2026.0    Tuesday      False
2 2026-07-13 14:30:00  2026.0     Monday      False
3                 NaT     NaN        NaN      False

--- Step 4: Filt

errors='coerce': Always use this when dealing with user-generated Excel files. If someone wrote "TBD" or left a cell messy, it won't break your script; it just turns into a missing timestamp (NaT), which you can drop or fill later.

format='mixed': Crucial for European vs. American date formats appearing in the same columns across different sheets.

Timezones: If you ever need to strip out timezone data to merge datasets safely, append .dt.tz_localize(None) to the end of your parsing line.